In [1]:
#imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBClassifier,XGBRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import ElasticNet


In [2]:
import xgboost
print(xgboost.__version__)

2.1.3


In [3]:
def read_data(data_path):
    #read data from csv file
    df = pd.read_csv(data_path)
    return df

In [4]:
def split_date(df):
    #split date to day, month, year
    df["day"] = df["Date"].str.split("/").str[0]
    df["month"] = df["Date"].str.split("/").str[1]
    df["year"] = df["Date"].str.split("/").str[2]
    df = df.drop(columns=["Date"])
    return df

In [5]:
def moving_average(df, window_size):
    #calculate moving average !!! this will skip the first 10 rows due to the window size
    df["MAVG"] = df["Last Close"].rolling(window=window_size).mean()
    return df

In [6]:
def pct_change(df):
    #calculate percentage change
    df["PCT"] = df["Last Close"].pct_change()
    return df

In [7]:
def create_lag_features(df, lag=1):
    # Create lag feature for 'Last Close' and 'PCT'
    df[f'Last Close Lag {lag}'] = df['Last Close'].shift(lag)
    df[f'PCT Lag {lag}'] = df['PCT'].shift(lag)
    return df

In [8]:
def OneHotEncode_Date(df):
    #one hot encode day, month, year
    df = pd.get_dummies(df, columns=["day", "month", "year"])
    return df

In [9]:
def split_train_test(df, test_size):
    #split data to train and test
    X = df.drop(columns=["PCT"])
    y = df["PCT"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    return X_train, X_test, y_train, y_test

In [10]:
def weekly_return(df):
    #calculate weekly return
    df["Weekly Return"] = df["Last Close"].pct_change(periods=5)
    return df

In [11]:
def train_xgb(df):
    # Preprocess the data
    df = OneHotEncode_Date(df)
    X_train, X_test, y_train, y_test = split_train_test(df, 0.2)

    # Parameter grid for hyperparameter tuning
    param_grid = {
        'n_estimators': [100, 200, 300],  # number of boosting rounds
        'learning_rate': [0.01, 0.05, 0.1],  # step size for each iteration
        'max_depth': [3, 5, 7],  # depth of the decision trees
        'subsample': [0.7, 0.8, 1.0],  # proportion of samples per tree
        'colsample_bytree': [0.7, 0.8, 1.0],  # proportion of features per tree
        'min_child_weight': [1, 3, 5],  # minimum sum of instance weight per child
        'gamma': [0, 0.1, 0.2],  # minimum loss reduction required to make a partition
    }

    model = XGBRegressor()
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Evaluate performance
    mae = mean_absolute_error(y_test, y_pred)
    print("XGBoost MAE with Hyperparameter Tuning: ", mae)
    

In [12]:
def train_RandomForestRegressor(df):
    X_train, X_test, y_train, y_test = split_train_test(df, 0.2)
    model = RandomForestRegressor()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Evaluate
    print("Random Forest Regressor MAE:", mean_absolute_error(y_test, y_pred))

In [13]:
def train_LinearRegression(df):
    X_train, X_test, y_train, y_test = split_train_test(df, 0.2)
    model = LinearRegression()
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Evaluate
    print("Linear Regression MAE:", mean_absolute_error(y_test, y_pred))

In [14]:
def train_Ridge(df):
    X_train, X_test, y_train, y_test = split_train_test(df, 0.2)
    ridge = Ridge()
    param_grid = {'alpha': np.logspace(-3, 3, 50)}
    grid = GridSearchCV(ridge, param_grid, cv=5, scoring='neg_mean_squared_error')
    grid.fit(X_train, y_train)

    print("Best alpha:", grid.best_params_)
    model = Ridge(alpha=grid.best_params_['alpha'])
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Evaluate
    print("Ridge Regression MAE:", mean_absolute_error(y_test, y_pred))

In [15]:

def train_ElasticNet(df, lag=1):
    X_train, X_test, y_train, y_test = split_train_test(df, 0.2)
    model = ElasticNet(alpha=1.0, l1_ratio=0.5)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    # Evaluate
    print("Elastic Net MAE:", mean_absolute_error(y_test, y_pred))

In [16]:

#read data
df = read_data("../data/historicalData_IE00B5BMR087_clean.csv")
df = split_date(df)
df = moving_average(df, 10)
df = weekly_return(df)
df = pct_change(df)
df = create_lag_features(df, 1)
df = df.dropna()
print(df.head())

train_RandomForestRegressor(df)
train_xgb(df)
train_LinearRegression(df)
train_Ridge(df)
train_ElasticNet(df)



    Open  High  Low  Last Close day month  year   MAVG  Weekly Return  \
9    220   220  220         220  01    18  2018  220.4       0.000000   
10   221   221  221         221  01    19  2018  220.6       0.000000   
11   221   221  221         221  01    22  2018  220.6       0.009132   
12   223   223  223         223  01    23  2018  220.6       0.013636   
13   221   221  221         221  01    24  2018  220.6       0.004545   

         PCT  Last Close Lag 1  PCT Lag 1  
9   0.000000             220.0   0.000000  
10  0.004545             220.0   0.000000  
11  0.000000             221.0   0.004545  
12  0.009050             221.0   0.000000  
13 -0.008969             223.0   0.009050  
Random Forest Regressor MAE: 0.0074831959713465655
XGBoost MAE with Hyperparameter Tuning:  0.005945562454426271
Linear Regression MAE: 0.0018821982345899125
Best alpha: {'alpha': 0.001}
Ridge Regression MAE: 0.0018814800431455874
Elastic Net MAE: 0.008160548660524155
